# 🚀 BiMamba-NAPM: Bidirectional Mamba with Normal-Anchored Prototype Memory
### Weakly-Supervised Video Anomaly Detection (WS-VAD)
**Tác giả:** Huỳnh Quang Vũ (MSSV: 2351010247)  
**Giảng viên hướng dẫn:** TS. Lê Viết Tuấn  
**Đơn vị:** Khoa Công nghệ Thông tin, Trường Đại học Mở TP. Hồ Chí Minh  

---
### 🎯 Mục tiêu hiệu năng:
- **UCF-Crime:** ROC-AUC >= 90.0%  
- **XD-Violence:** Average Precision (AP) > 87.0%  
- **Phương thức:** Huấn luyện từ đầu (Train from Scratch 100%), không mượn checkpoint warm-start của DSANet.

In [ ]:
# ========================= 1. CLONE & CÀI ĐẶT MÔI TRƯỜNG =========================
import os, sys, shutil, subprocess, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "BiMamba-NAPM"
ARTIFACTS = WORK / "bimamba_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Tự động Clone hoặc Pull code mới nhất từ GitHub
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Sharvuz/BiMamba-NAPM.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=False)

os.chdir(str(REPO))

# Cài đặt OpenAI CLIP và các thư viện cần thiết
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "scikit-learn", "pandas", "git+https://github.com/openai/CLIP.git"
], check=True)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"GPU Count: {torch.cuda.device_count()}")


## ⚙️ PHẦN 1: HUẤN LUYỆN TRÊN TẬP DỮ LIỆU UCF-CRIME

In [ ]:
# ========================= 2. TỰ ĐỘNG QUÉT VÀ KHỚP NỐI ĐƯỜNG DẪN DỮ LIỆU UCF =========================
from collections import defaultdict
from pathlib import PurePosixPath
import pandas as pd
import numpy as np

DATASET = "ucf"
csv_specs = [
    ("list/ucf_CLIP_rgb.csv", "ucf_train.csv"),
    ("list/ucf_CLIP_rgbtest.csv", "ucf_test.csv"),
]
zip_hints = ["ucfclipfeatures", "ucf"]

def basename(p):
    return PurePosixPath(str(p).replace("\\", "/")).name

required_names = set()
for csv_rel, _ in csv_specs:
    frame = pd.read_csv(REPO / csv_rel)
    required_names.update(basename(p) for p in frame["path"])

feature_roots = [INPUT, WORK]

# Tự động giải nén nếu dữ liệu nằm trong file ZIP
archives = [p for p in INPUT.rglob("*.zip") if any(h in p.stem.lower() for h in zip_hints)]
if archives:
    extract_root = WORK / f"features_{DATASET}"
    extract_root.mkdir(parents=True, exist_ok=True)
    for arc in archives:
        dest = (extract_root / arc.stem).resolve()
        if not dest.exists() or not any(dest.iterdir()):
            dest.mkdir(parents=True, exist_ok=True)
            print(f"Đang giải nén {arc.name}...")
            with zipfile.ZipFile(arc) as z:
                z.extractall(dest)
            print(f"Đã giải nén: {arc.name}")
    feature_roots.append(extract_root)

# Quét radar tìm tất cả file .npy
index = defaultdict(list)
for root in feature_roots:
    for p in root.rglob("*.npy"):
        if p.name in required_names:
            index[p.name].append(p)

def rewrite_csv(csv_rel, output_name):
    frame = pd.read_csv(REPO / csv_rel)
    resolved, missing = [], []
    for original in frame["path"].astype(str):
        bname = basename(original)
        matches = index.get(bname, [])
        if not matches:
            missing.append(bname)
            resolved.append("")
        else:
            resolved.append(str(matches[0]))
    if missing:
        raise FileNotFoundError(f"{csv_rel}: thiếu {len(missing)} files .npy trong /kaggle/input! Ví dụ: {missing[:5]}")
    output = ARTIFACTS / output_name
    frame["path"] = resolved
    frame.to_csv(output, index=False)
    print(f"Đã đồng bộ {output_name}: {len(frame)} files sẵn sàng.")
    return output

train_csv_ucf = rewrite_csv(*csv_specs[0])
test_csv_ucf = rewrite_csv(*csv_specs[1])
print("✅ Bản đồ dữ liệu UCF-Crime đã sẵn sàng 100%!")


In [ ]:
# ========================= 3. HUẤN LUYỆN BIMAMBA-NAPM (UCF-CRIME) =========================
MAX_EPOCHS = 15
UCF_BATCH_SIZE = 64
LR = 5e-5
SSM_TYPE = "s4d"

dataset_dir = ARTIFACTS / "ucf"
dataset_dir.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable, "-u", "src/ucf_train.py",
    "--dataset", "ucf",
    "--train_list", str(train_csv_ucf),
    "--test_list", str(test_csv_ucf),
    "--ssm_type", SSM_TYPE,
    "--num_layers", "2",
    "--num_prototypes", "16",
    "--batch_size", str(UCF_BATCH_SIZE),
    "--lr", str(LR),
    "--max_epoch", str(MAX_EPOCHS),
    "--loss_napm_weight", "0.5",
    "--loss_align_weight", "0.5",
    "--save_dir", str(dataset_dir)
]

print("🚀 Bắt đầu huấn luyện BiMamba-NAPM trên UCF-Crime...")
subprocess.run(train_cmd, check=True)


In [ ]:
# ========================= 4. ĐÁNH GIÁ (UCF-CRIME) =========================
best_model_path = ARTIFACTS / "ucf" / "bimamba_napm_ucf_best.pth"

eval_cmd = [
    sys.executable, "-u", "src/ucf_test.py",
    "--dataset", "ucf",
    "--test_list", str(test_csv_ucf),
    "--ssm_type", SSM_TYPE,
    "--model_path", str(best_model_path)
]
print("🔍 Đang đánh giá model tốt nhất...")
subprocess.run(eval_cmd, check=True)


## ⚙️ PHẦN 2: HUẤN LUYỆN TRÊN TẬP DỮ LIỆU XD-VIOLENCE

In [ ]:
# ========================= 5. TỰ ĐỘNG QUÉT VÀ KHỚP NỐI ĐƯỜNG DẪN DỮ LIỆU XD-VIOLENCE =========================
DATASET_XD = "xd"
csv_specs_xd = [
    ("list/xd_CLIP_rgb.csv", "xd_train.csv"),
    ("list/xd_CLIP_rgbtest.csv", "xd_test.csv"),
]

all_npy_paths = list(INPUT.rglob("*.npy")) + list(WORK.rglob("*.npy"))
file_map = {p.name: p for p in all_npy_paths}

def rewrite_csv_xd(csv_rel, output_name):
    input_csv = REPO / csv_rel
    output = ARTIFACTS / output_name
    frame = pd.read_csv(input_csv)
    resolved, missing = [], []
    for p in frame["path"].astype(str):
        bname = Path(p).name
        # Thử các định dạng tên của Kaggle (# thành +, _, hoặc bỏ dấu)
        candidates = [bname, bname.replace("#", "+"), bname.replace("#", "_"), bname.replace("#", "")]
        found = None
        for c in candidates:
            if c in file_map:
                found = str(file_map[c])
                break
        if found:
            resolved.append(found)
        else:
            missing.append(bname)
    if missing:
        raise FileNotFoundError(f"{csv_rel}: thiếu {len(missing)} files .npy trong /kaggle/input! Ví dụ: {missing[:5]}")
    frame["path"] = resolved
    frame.to_csv(output, index=False)
    print(f"Đã đồng bộ {output_name}: {len(frame)} files sẵn sàng.")
    return output

train_csv_xd = rewrite_csv_xd(*csv_specs_xd[0])
test_csv_xd = rewrite_csv_xd(*csv_specs_xd[1])
print("✅ Bản đồ dữ liệu XD-Violence đã sẵn sàng 100%!")


In [ ]:
# ========================= 6. HUẤN LUYỆN BIMAMBA-NAPM (XD-VIOLENCE) =========================
MAX_EPOCHS_XD = 20
XD_BATCH_SIZE = 64
LR_XD = 1e-4

dataset_xd_dir = ARTIFACTS / "xd"
dataset_xd_dir.mkdir(parents=True, exist_ok=True)

train_xd_cmd = [
    sys.executable, "-u", "src/xd_train.py",
    "--dataset", "xd",
    "--train_list", str(train_csv_xd),
    "--test_list", str(test_csv_xd),
    "--ssm_type", SSM_TYPE,
    "--num_layers", "2",
    "--num_prototypes", "16",
    "--batch_size", str(XD_BATCH_SIZE),
    "--lr", str(LR_XD),
    "--max_epoch", str(MAX_EPOCHS_XD),
    "--loss_napm_weight", "0.5",
    "--loss_align_weight", "0.5",
    "--save_dir", str(dataset_xd_dir)
]

print("🚀 Bắt đầu huấn luyện BiMamba-NAPM trên XD-Violence...")
subprocess.run(train_xd_cmd, check=True)


In [ ]:
# ========================= 7. ĐÁNH GIÁ (XD-VIOLENCE) =========================
best_model_xd = ARTIFACTS / "xd" / "bimamba_napm_xd_best.pth"

eval_xd_cmd = [
    sys.executable, "-u", "src/xd_test.py",
    "--dataset", "xd",
    "--test_list", str(test_csv_xd),
    "--ssm_type", SSM_TYPE,
    "--model_path", str(best_model_xd)
]
print("🔍 Đang đánh giá model XD-Violence tốt nhất...")
subprocess.run(eval_xd_cmd, check=True)
